# **Getting the data by upload it manually since it's small**
www.kaggle.com/datasets/sinjoysaha/sales-analysis-dataset


# **Prepare The Data** **ℾ**

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import joblib

# Getting the data
df = pd.read_csv("/kaggle/input/datasets/ouicifarouk/data-electronics/ready_electonic_sales_data.csv")

# transform the date column into pandas date-type
df['Date'] = pd.to_datetime(df['Date'])
df['Week'] = df['Date'].dt.isocalendar().week

# combining the rows based on some features
df_new = df.groupby(["Product","Year","Week","Day"]).agg(
    Quantity_Ordered=("Quantity Ordered", "sum"), # here we sum the quantity oredered
    Price_Each = ("Price Each","mean"),
    ).reset_index() # to not return the columns index

print(df_new)

# Making the months cyclical
df_new["Week_sin"] = np.sin(2.0 * np.pi * df_new["Week"] / 52.0)
df_new["Week_cos"] = np.cos(2.0 * np.pi * df_new["Week"] / 52.0)

# Sorting the data 
df_new = df_new.sort_values(by=["Year", "Week", "Day"])

# Relating the Sales between the weeks
df_new['Qty_Lag_1'] = df_new.groupby('Product')['Quantity_Ordered'].shift(1)
df_new['Qty_Rolling_3'] = df_new.groupby('Product')['Quantity_Ordered'].transform(
    lambda x: x.shift(1).rolling(3).mean()
)
df_new = df_new.fillna(0)

# Slicing 10% of the data to test the model with it 
df_test = df_new.tail(int(len(df_new) * 0.1))
# Dropping the data for the real test to make sure that the model don't see it ever until we use it to predict
df_new = df_new.drop(df_test.index) 

# Saving the dataframe 
df_new.to_csv("electronic-store-sales_cleaned.csv",index=False)

# we split the data based on conditions to overcome the issue of future leakage
train = df_new[(df_new["Year"] == 2019) & (df_new["Week"] < 38)]
test = df_new[(df_new["Week"] >= 38) & (df_new["Year"] == 2019) | (df_new["Year"] == 2020)]

# Spliting the data to train and test
X_train = train.drop("Quantity_Ordered",axis=1).copy()
y_train = train["Quantity_Ordered"].copy()

X_test = test.drop("Quantity_Ordered",axis=1).copy()
y_test = test["Quantity_Ordered"].copy()


print("The Data Has been splited")

print(X_train.head())
print(X_train.shape)
print(X_train["Product"].value_counts())

# **Applying the techniques** ∮

In [ ]:
import time
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import FunctionTransformer,StandardScaler,OneHotEncoder
from sklearn.model_selection import TimeSeriesSplit,cross_val_score
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor


# Rounding the decimals
X_train["Price_Each"] = X_train["Price_Each"].round()
X_test["Price_Each"] = X_test["Price_Each"].round()

print(f"The X_train shape is : {X_train.shape}")

# Selecting the columns for each transform
one_hot_col = ["Product"]
log_col = ["Price_Each"]

# Defining the logarithmic transformer 
log_function = FunctionTransformer(np.log1p)

# One-hot pipeline
one_hot_pipeline = Pipeline([
    ("onehot",OneHotEncoder(sparse_output=False,handle_unknown="ignore")),
])

# Log pipeline to apply it to the log_col and scale it
log_pipeline = Pipeline([
    ("log_transformer", log_function), 
     ("scaler", StandardScaler())
])

# The pre-pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("one_hot", one_hot_pipeline, one_hot_col), # Applying the transformers we defined
        ("log", log_pipeline, log_col),
    ],
    remainder='passthrough'
)

# The model (which is a ready pipeline that transform the features and predict)
linear_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "regressor",
            LinearRegression(),
        ), 
    ]
)

model = TransformedTargetRegressor(
    regressor=linear_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

# Training
t = time.time() # Counting the time that takes the model to train
model.fit(X_train,y_train)
t_a = time.time()
print(f'the model was trained on :  {(t_a - t):.4f} s')

# Testing the model
y_pred = model.predict(X_test)
tscv = TimeSeriesSplit(n_splits=5,test_size=100)
scores = cross_val_score(model, X_train, y_train, cv=tscv, scoring='r2') # Using the cross-validation to see the model performance
print(f"Mean CV R²: {scores.mean():.4f} (+/- {scores.std():.4f})")


# The measures to rate the model preformance
mse_a = mean_squared_error(y_test, y_pred)
rmse_a= np.sqrt(mse_a)
mae_a = mean_absolute_error(y_test, y_pred)
r2_a = r2_score(y_test, y_pred)

# Model prediction on the train inputs
y_t_pred = model.predict(X_train)

# The measures for model performance on the train data
r2_t = r2_score(y_train, y_t_pred)
mae_t = mean_absolute_error(y_train, y_t_pred)
rmse_t = np.sqrt(mean_squared_error(y_train, y_t_pred))

print("\n=== Train vs Test Performance ===")
print(f"Train R² : {r2_t:.4f}  |  Test R² : {r2_a:.4f}")
print(f"Train MAE: {mae_t:.4f}  |  Test MAE: {mae_a:.4f}")
print(f"Train RMSE: {rmse_t:.4f} |  Test RMSE: {rmse_a:.4f}")


# Testing the model with the new data

X = df_test.drop(columns=["Quantity_Ordered"])
y = df_test["Quantity_Ordered"]

# Model prediction in the real-test data
prediction = model.predict(X)

r2_r = r2_score(y, prediction)
mae_r = mean_absolute_error(y, prediction)
rmse_r = np.sqrt(mean_squared_error(y, prediction))

print("\n=== Train vs Test Performance ===")
print(f"The R² is : {r2_r}")
print(f"The MAE is : {mae_r}")
print(f"The RMSE is : {rmse_r}")


# Saving the model

joblib.dump(model,"DemandMind.pkl")

In [ ]:
import time
import numpy as np
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor

# 1. Define the raw XGBoost Pipeline
raw_xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(random_state=42, n_jobs=-1))
])

# 2. Wrap the Pipeline with TransformedTargetRegressor to apply log1p on target (y)
xgb_target_model = TransformedTargetRegressor(
    regressor=raw_xgb_pipeline,
    func=np.log1p,        # Applies log(y + 1) during training
    inverse_func=np.expm1 # Reverts predictions back to the original scale
)

# 3. Define the hyperparameter search space
# Note: Double 'regressor__' prefix is required due to wrapping Pipeline inside TransformedTargetRegressor
# Reduced max_depth values (2-5) to prevent severe overfitting
parms_distrubtion = {
    "regressor__regressor__max_depth": [2, 3, 4, 5],
    "regressor__regressor__n_estimators": [100, 150, 200],
    "regressor__regressor__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "regressor__regressor__subsample": [0.6, 0.8, 1.0],
    "regressor__regressor__colsample_bytree": [0.6, 0.8, 1.0],
}

# 4. Perform Random Search with Successive Halving using TimeSeriesSplit
print("Start The Random Search....")
random_search = RandomizedSearchCV(
    estimator=xgb_target_model,
    param_distributions=parms_distrubtion,
    n_iter=20, 
    cv=tscv,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)
t_s = time.time()
random_search.fit(X_train, y_train)
t_e = time.time()
print("Finished!")
print(f"The best parameters are: {random_search.best_params_} found in: {(t_e - t_s):.4f} s")

# 5. Extract the best parameters and construct a fine-tuning parameter grid
best_depth = random_search.best_params_["regressor__regressor__max_depth"]
best_est = random_search.best_params_["regressor__regressor__n_estimators"]

param_grid_fine = {
    "regressor__regressor__n_estimators": [
        max(30, best_est - 20),
        best_est,
        best_est + 20,
    ],
    "regressor__regressor__max_depth": [max(1, best_depth - 1), best_depth, best_depth + 1],
    "regressor__regressor__learning_rate": [
        random_search.best_params_["regressor__regressor__learning_rate"]
    ],
    "regressor__regressor__subsample": [
        random_search.best_params_["regressor__regressor__subsample"]
    ],
    "regressor__regressor__colsample_bytree": [
        random_search.best_params_["regressor__regressor__colsample_bytree"]
    ],
}

# 6. Perform Fine-Tuning using Grid Search
print("Starting the GridSearch....")
grid_search = GridSearchCV(
    estimator=xgb_target_model,
    param_grid=param_grid_fine,
    cv=tscv,
    scoring='r2',
    n_jobs=-1
)

time_st = time.time()
grid_search.fit(X_train, y_train)
time_en = time.time()

# Get the final optimized model
best_model = grid_search.best_estimator_
print(f"The model was fine-tuned successfully in: {(time_en - time_st):.4f} s")

# 7. Evaluate Model Performance on CV, Train, and Test Sets
scores = cross_val_score(best_model, X_train, y_train, cv=tscv, scoring='r2')
print(f"Mean CV R²: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Model predictions on test set (automatically exponentiated back to original scale)
model_pred = best_model.predict(X_test)

mse = mean_squared_error(y_test, model_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, model_pred)
r2 = r2_score(y_test, model_pred)

# Model predictions on training set for overfitting evaluation
y_train_pred = best_model.predict(X_train)

r2_train = r2_score(y_train, y_train_pred)
mae_train = mean_absolute_error(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))

print("\n=== XGBoost Train vs Test Performance ===")
print(f"Train R² : {r2_train:.4f}  |  Test R² : {r2:.4f}")
print(f"Train MAE: {mae_train:.4f}  |  Test MAE: {mae:.4f}")
print(f"Train RMSE: {rmse_train:.4f} |  Test RMSE: {rmse:.4f}")

# 8. Evaluate on the Holdout Real-World Test Data (df_test)
X_holdout = df_test.drop(columns=["Quantity_Ordered"])
y_holdout = df_test["Quantity_Ordered"]

holdout_pred = best_model.predict(X_holdout)

r2_holdout = r2_score(y_holdout, holdout_pred)
mae_holdout = mean_absolute_error(y_holdout, holdout_pred)
rmse_holdout = np.sqrt(mean_squared_error(y_holdout, holdout_pred))

print("\n=== Holdout Real-World Test Performance ===")
print(f"Holdout R²  : {r2_holdout:.4f}")
print(f"Holdout MAE : {mae_holdout:.4f}")
print(f"Holdout RMSE: {rmse_holdout:.4f}")